# Profiling Messy CSVs with Dataprof

In data engineering, you often receive CSV files that contain mixed data types, missing values, and corrupted text. This exercise shows how to use `dataprof` to swiftly detect and summarize these issues.

In [1]:
# Run this cell first to install dependencies in Colab
!pip install pandas dataprof==0.7.0

error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try apt install
    python3-xyz, where xyz is the package you are trying to
    install.
    
    If you wish to install a non-Debian-packaged Python package,
    create a virtual environment using python3 -m venv path/to/venv.
    Then use path/to/venv/bin/python and path/to/venv/bin/pip. Make
    sure you have python3-full installed.
    
    If you wish to install a non-Debian packaged Python application,
    it may be easiest to use pipx install xyz, which will manage a
    virtual environment for you. Make sure you have pipx installed.
    
    See /usr/share/doc/python3.12/README.venv for more information.

note: If you believe this is a mistake, please contact your Python installation or OS distribution provider. You can override this, at the risk of breaking your Python installation or OS, by passing --break-system-packages.
hint: See PEP 668 for the detai

## Generate Messy Dummy Data

Let's create a terribly formatted CSV to simulate wild, unclean data. Notice the arbitrary date formats, missing objects, and mixed numeric/text fields.

In [2]:
import pandas as pd
import numpy as np

# Build the corrupted dataset elements
messy_data = {
    'user_id': ['101', '102', '103', 'UNKNOWN', '105', '106', '107', None, '109', '110'],
    'age': [25, 30, 'thirty-five', 40, np.nan, 22, -5, 29, 31, 899],
    'signup_date': ['2023-01-01', '2023-01-02', 'not_a_date', '2023/15/01', '2023-01-05', '01-06-2023', pd.NaT, '2023-01-08', '20230109', '2023-01-10'],
    'score': [8.5, 9.0, np.nan, 7.5, 8.0, 9.5, 6.0, 7.0, np.nan, 8.2]
}

df_messy = pd.DataFrame(messy_data)
df_messy.to_csv("messy_data.csv", index=False)
print("Created 'messy_data.csv'!")

display(df_messy)

Created 'messy_data.csv'!


,user_id,age,signup_date,score
0,101,25,2023-01-01,8.5
1,102,30,2023-01-02,9.0
2,103,thirty-five,not_a_date,NaN
3,UNKNOWN,40,2023/15/01,7.5
4,105,NaN,2023-01-05,8.0
5,106,22,01-06-2023,9.5
6,107,-5,NaT,6.0
7,NaN,29,2023-01-08,7.0
8,109,31,20230109,NaN
9,110,899,2023-01-10,8.2


## Profile the Data

We'll load the literal string versions of this CSV (which is how pandas usually falls back on messy data without schema pre-validation) and then use `dataprof` to expose the anomalies.

In [3]:
import dataprof as dp

# 1. Load the data exactly as it appears in the wild
df_raw = pd.read_csv("messy_data.csv", dtype=str)

# 2. Hand it off to dataprof to crunch the underlying anomalies
# metrics= is explicit here so all packs run — good to dogfood pattern detection
report = dp.profile(df_raw, metrics=["schema", "statistics", "patterns", "quality"])

# 3. Extract key metrics
print("--- DATA PROFILE SUMMARY ---")
print(f"Total Rows Diagnosed: {report.rows}")
print(f"Source Exhausted: {report.source_exhausted}")
print(f"Throughput: {report.throughput:.0f} rows/sec" if report.throughput else "Throughput: N/A")
print(f"Memory Peak: {report.memory_peak_mb:.1f} MB" if report.memory_peak_mb else "Memory Peak: N/A")

print("\n--- CRITICAL COLUMN INSIGHTS ---")
# column_profiles is a dict {name: ColumnProfile} — iterate its values
for col_profile in report.column_profiles.values():
    print(f"\nColumn: '{col_profile.name}'")
    print(f"  - Type: {col_profile.data_type}")
    print(f"  - Missing Value Rate: {col_profile.null_count} out of {report.rows} ({col_profile.null_percentage:.0f}%)")
    print(f"  - Unique Values: {col_profile.unique_count} ({col_profile.uniqueness_ratio:.0%})")

    # Numeric stats
    if col_profile.min is not None:
        print(f"  - Range: {col_profile.min} to {col_profile.max}")
    if col_profile.mean is not None:
        print(f"  - Mean: {col_profile.mean:.2f}  Std: {col_profile.std_dev:.2f}")

    # Text-length stats (string columns)
    if col_profile.avg_length is not None:
        print(
            f"  - String lengths: min={col_profile.min_length}, "
            f"max={col_profile.max_length}, avg={col_profile.avg_length:.1f}"
        )

    # Detected value patterns (e.g. date, email, IP, …)
    if col_profile.patterns:
        best = max(col_profile.patterns, key=lambda p: p.match_count)
        print(f"  - Top pattern: {best.name} ({best.match_percentage:.0f}% of values)")
    else:
        print(f"  - No patterns detected")

# 4. Quick statistical summary — like DataFrame.describe() but for the profile
print("\n--- STATISTICAL SUMMARY (describe) ---")
print(report.describe())

# 5. Quality score at a glance
print("\n--- QUALITY SUMMARY ---")
print(report.quality_summary())

# 6. Write full schema out for automated CI/CD checks next time
# save() supports .json, .csv, and .parquet; chains via return self
print("\nSaving report to JSON and CSV for further inspection...")
report.save("messy_report.json").save("messy_report.csv")
print("Done.")

--- DATA PROFILE SUMMARY ---
Total Rows Diagnosed: 10
Source Exhausted: True
Throughput: 2000 rows/sec
Memory Peak: N/A

--- CRITICAL COLUMN INSIGHTS ---

Column: 'score'
  - Type: float
  - Missing Value Rate: 2 out of 10 (20%)
  - Unique Values: 8 (80%)
  - Range: 6.0 to 9.5
  - Mean: 7.96  Std: 1.12
  - No patterns detected

Column: 'signup_date'
  - Type: string
  - Missing Value Rate: 1 out of 10 (10%)
  - Unique Values: 9 (90%)
  - String lengths: min=8, max=10, avg=9.8
  - No patterns detected

Column: 'age'
  - Type: float
  - Missing Value Rate: 1 out of 10 (10%)
  - Unique Values: 9 (90%)
  - Range: -5.0 to 899.0
  - Mean: 133.88  Std: 309.44
  - No patterns detected

Column: 'user_id'
  - Type: float
  - Missing Value Rate: 1 out of 10 (10%)
  - Unique Values: 9 (90%)
  - Range: 101.0 to 110.0
  - Mean: 105.38  Std: 3.25
  - No patterns detected

--- STATISTICAL SUMMARY (describe) ---
               score  signup_date       age   user_id
count        10.0000      10.0000   1

## Dogfooding Notes — dataprof 0.7.0

Observations from running this notebook against the library:

- **Pattern detection threshold**: `signup_date` mixes 4+ date formats (`ISO`, `DD-MM-YYYY`, `YYYYMMDD`, invalid) but no pattern fires. With only 10 rows, no single format reaches the match threshold. On larger datasets this would correctly surface the dominant format.
- **`memory_peak_mb` is `None` for DataFrame inputs**: only populated when profiling files via the incremental engine. The `if` guard above handles it safely.
- **`user_id` inferred as `float`**: correct — `UNKNOWN` becomes null, remaining values are numeric. dataprof's type inference reads the cleaned signal, not the raw string.
- **`accuracy: 100.0`** despite `age` having `-5` and `899` as outliers — the accuracy dimension checks range violations against defined constraints, not statistical outliers. Combine with manual inspection of `min`/`max` for anomaly hunting.
- **`completeness: 50.0`** — driven by the column-level null rates across all 4 columns (1+2+1+1 nulls / 40 total cells = 87.5% non-null… score reflects weighted ISO 25012 formula, not raw null rate).